In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
%matplotlib widget

In [ ]:
ts = -1
with xr.open_dataset('hydraulic.snapshot.nc') as ds:
    dd = ds.isel(yt=1, yu=1, Time=ts)
    fig, axs = plt.subplots(3, 1, height_ratios=[1, 5, 5], sharex=True, layout='constrained')
    pc = axs[1].pcolormesh(dd.xt, dd.zt, dd.temp, cmap='RdBu_r', clim=np.array([10, 11]))
    fig.colorbar(pc)
    axs[2].pcolormesh(dd.xu, dd.zt, dd.u, cmap='RdBu_r', clim=np.array([-1, 1]))
    #axs[1].contour(dd.yt, dd.zt, dd.temp)
    #axs[0].set_xlim(20, 0)
    fig, axs = plt.subplots()
    pc = axs.plot( ds.xt, ds.psi[:, 2,].T)


In [ ]:
fig, axs = plt.subplots(4, 5, sharex=True, sharey=True, layout='constrained')
tds = np.arange(0, 100, 5)
with xr.open_dataset('hydraulic.snapshot.nc') as ds:
    display(ds)
    for nn, td in enumerate(tds):
        dd = ds.isel(yt=1, yu=1, Time=td)
        x0 = np.mean(ds.xu)
        pc = axs.flat[nn].pcolormesh(dd.xu-x0, dd.zt, dd.temp, cmap='RdBu_r', clim=[10, 11]) #, clim=np.array([-1, 1])/5)
        axs[0,0].set_xlim([-30, 30])

In [ ]:
fig, ax = plt.subplots()
ax.pcolormesh(dd.temp.values)

In [ ]:
with xr.open_dataset('hydraulic.snapshot.nc') as ds:
    fig, ax = plt.subplots()
    ax.plot(ds.xt, ds.dxt)
    ax.plot(ds.xu, ds.dxu)


In [ ]:
with xr.open_dataset('hydraulic.snapshot.nc') as ds:
    fig, ax = plt.subplots()
    ax.plot(ds.dxt, ds.xt*0, '.r')
    ax.plot(ds.dxu, ds.xu*0, '.')


In [ ]:
fig, ax = plt.subplots()
ax.plot(ds.xt, ds.dxt)
ax.plot(ds.xu, ds.dxu)

## Momentum Balance:


### du/dt

In [ ]:

time=20
with xr.open_dataset('hydraulic.snapshot.nc') as ds:
    ds = ds.isel(yt=1, yu=1)
    dudt = ds.u.diff(dim='Time', label='upper') / 7200

    fig, ax = plt.subplots()
    ax.pcolormesh(dudt.isel(Time=time), norm=norm, cmap='RdBu_r')

### Momentum budget

In [ ]:
from matplotlib.colors import Normalize
norm = Normalize(vmin=-20e-6, vmax=20e-6)
time = 20
with xr.open_dataset('hydraulic.snapshot.nc') as ds:
    ds = ds.isel(yt=1, yu=1, Time=time)

    
    dpdx = (ds.p_hydro + ds.psi).differentiate('xt') / 1000
    hori = 4 * ds.u.differentiate('xu').differentiate('xu') / 1000 / 1000
    vert = 4e-4 * ds.u.differentiate('zt').differentiate('zt').interp(xu=ds.xt.values).values
    
    ududx = ds.u * ds.u.differentiate('xu') / 1000
    wdudz = +ds.w.interp(xt=ds.xu.values).values * ds.u.differentiate('zt')

    fig, axs = plt.subplots(3, 2, sharex=True, sharey=True)
    axs[0, 0].pcolormesh(ds.xt, ds.zt, -dpdx,  norm=norm, cmap='RdBu_r')
    axs[0, 0].set_title('-dp/dx')
    axs[0, 1].pcolormesh(ds.xt, ds.zt,ududx,  norm=norm, cmap='RdBu_r')
    axs[0, 1].set_title('u du/dx')
    axs[1, 0].pcolormesh(ds.xt, ds.zt,vert+hori,  norm=norm, cmap='RdBu_r')
    axs[1, 0].set_title('vertical stress divergence')
    axs[1, 1].pcolormesh(ds.xt, ds.zt,wdudz,  norm=norm, cmap='RdBu_r')
    axs[1, 1].set_title('w du/dz')
    axs[2, 0].pcolormesh(ds.xt, ds.zt,(-dpdx + vert).rolling(xt=4).mean() ,  norm=norm, cmap='RdBu_r')
    axs[2, 1].pcolormesh(ds.xt, ds.zt, (ududx + wdudz).rolling(xu=4).mean(),  norm=norm, cmap='RdBu_r')

    axs[0, 0].set_xlim([-30, 30])
    

### u du/dx

In [ ]:
with xr.open_dataset('hydraulic.snapshot.nc') as ds:
    ds = ds.isel(yt=1, yu=1)

    ududx = ds.u * ds.u.differentiate('xu') / 1000
    
    wdudz = +ds.w.interp(xt=ds.xu.values).values * ds.u.differentiate('zt')
    #wdudz = +ds.w.values * ds.u.differentiate('zt')
    
    fig, ax = plt.subplots()
    ax.pcolormesh((wdudz + ududx).isel(Time=time).rolling(xu=1).mean(), norm=norm, cmap='RdBu_r')


## stresses

In [ ]:
with xr.open_dataset('hydraulic.snapshot.nc') as ds:
    ds = ds.isel(yt=1, yu=1)

    hori = 4 * ds.u.differentiate('xu').differentiate('xu') / 1000 / 1000
    fig, ax = plt.subplots()
    ax.pcolormesh(hori.isel(Time=time), norm=norm, cmap='RdBu_r')


    vert = 4e-4 * ds.u.differentiate('zt').differentiate('zt') 
    fig, ax = plt.subplots()
    ax.pcolormesh(vert.isel(Time=time), norm=norm, cmap='RdBu_r')

### Layer view

In [ ]:
with xr.open_dataset('hydraulic.snapshot.nc') as ds:
    ds = ds.isel(yt=1, yu=1, Time=time)
    ds['layer'] = xr.where(ds.temp<=10.1, 1, 0)
    ds['h2'] = ds.layer.sum(dim='zt') * 200/90
    ds['ubottom'] = xr.where(ds.temp.values<=12, ds.u, 0)

In [ ]:
fig, ax = plt.subplots()
ax.plot(ds.xt, ds.h2 -ds.ht)
ax.pcolormesh(ds.xt, ds.zt, ds['layer'])

In [ ]:
fig, ax = plt.subplots()
ax.plot(ds.xt, ds.ubottom.sum(dim='zt'))

In [ ]:
fig, ax = plt.subplots()
ax.pcolormesh(ds.ubottom, vmin=-0.6, vmax=0.6)

## Test continuity

In [ ]:
with xr.open_dataset('hydraulic.snapshot.nc') as ds:
    ds = ds.isel(yt=1, yu=1, Time=time)
    topi = 50
    lefti = 50
    righti = 100
    left = (ds.u[0:topi, lefti]*200/90).sum(dim='zt')
    print(left)
    right = (ds.u[0:topi, righti]*200/90).sum(dim='zt')
    print(right)
    top = (ds.w[topi-1, lefti:righti] * ds.dxt[lefti:righti]).sum(dim='xt')
    print(top)
    print(left-top-right)

In [ ]:
ds.w